In [1]:
import torch
from transformers import AutoModel

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
model = AutoModel.from_pretrained(
    "antichronology/orthrus-small-6-track",
    trust_remote_code=True
).to(device).eval()

In [3]:
import genome_kit as gk 
from genome_kit import Genome, Interval
import torch
from orthrus.gk_utils import find_transcript_by_gene_name, create_six_track_encoding

/home/ray/projects/compbio/Orthrus/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [4]:
genome = Genome("gencode.v29")

In [5]:
transcripts = find_transcript_by_gene_name(genome, 'BRCA2')
t = transcripts[0]
t

<Transcript ENST00000380152.7 of BRCA2>

In [6]:
sixt = torch.tensor(create_six_track_encoding(t, genome), dtype=torch.float32)
sixt = torch.unsqueeze(sixt, 0) # (1, 6, L)
print(sixt)
print(sixt.shape)

device = torch.device("cuda")
model = model.to(device)
sixt = sixt.to(device)
lengths = torch.tensor([sixt.shape[1]]).to(device)


tensor([[[0., 0., 0.,  ..., 1., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 1.]]])
torch.Size([1, 6, 11986])


In [7]:
sixt = torch.transpose(sixt, 1, 2)
sixt.shape

torch.Size([1, 11986, 6])

In [8]:

with torch.no_grad():
    embedding = model.representation(sixt, lengths, channel_last=True)
embedding.shape

torch.Size([1, 256])